In [ ]:
# ============================================================
# GOOGLE COLAB SETUP — run this cell first when using Colab
# ============================================================
import sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_PATH = '/content/drive/MyDrive/Factor-Research'

    if not os.path.exists(REPO_PATH):
        print('Cloning repository to Google Drive...')
        os.system(f'git clone https://github.com/mbrennan5/Factor-Research.git {REPO_PATH}')
    else:
        print(f'Repository found at {REPO_PATH}')

    print('Installing packages...')
    os.system('pip install -q lightgbm xgboost optuna plotly tqdm yfinance alpaca-py pyarrow')

    NOTEBOOKS_DIR = os.path.join(REPO_PATH, 'notebooks')
    os.chdir(NOTEBOOKS_DIR)
    print(f'Working directory set to: {os.getcwd()}')
else:
    print('Running locally — no Colab setup needed.')


# Data Preparation and Feature Engineering

This notebook is the **first step** in the quantitative research pipeline. It handles the loading, cleaning, and preprocessing of raw minute-level stock data and Barra factor data.

## **Objective**

1.  **Load Raw Data**: Ingest minute-level OHLCV (Open, High, Low, Close, Volume) data and Barra factor data.
2.  **Data Cleaning**: Handle missing values, sort data by time, and ensure data integrity.
3.  **Feature Engineering**: Calculate key financial metrics that will be used as base features for alpha factor generation.
4.  **Save Processed Data**: Store the cleaned and engineered data in a structured format for downstream use.

## **Key Features Generated**

-   **Daily Return Rate**: Core metric for stock performance.
-   **Daily Turnover Rate**: Liquidity indicator.
-   **VWAP (Volume-Weighted Average Price)**: Price metric adjusted for trading volume.
-   **Barra Size Factor**: Market capitalization-based risk factor.


In [ ]:
# === 1. Import Libraries & Configure Alpaca Credentials ===

import pandas as pd
import numpy as np
import os
import warnings
from tqdm import tqdm
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame

warnings.filterwarnings('ignore')

# Credentials: read from env vars; fall back to the values below.
# In Colab set these as Secrets (key icon) so they are never hard-coded.
API_KEY_ID     = os.environ.get('ALPACA_API_KEY',    'AKXDM3YDOKE9EJDBQWZC')
API_SECRET_KEY = os.environ.get('ALPACA_SECRET_KEY', 'duaAQYetNb5nJ5gSRXgbvmjU0cEkSEJGjZiupMiE')
BASE_URL       = 'https://paper-api.alpaca.markets'  # paper trading endpoint

client = StockHistoricalDataClient(API_KEY_ID, API_SECRET_KEY)

# --- Universe of liquid US stocks (edit to add/remove symbols) ---
SYMBOLS = [
    'AAPL', 'MSFT', 'AMZN', 'GOOGL', 'META', 'NVDA', 'TSLA', 'JPM',
    'JNJ',  'V',    'PG',   'UNH',   'HD',   'MA',   'BAC',  'ADBE',
    'NFLX', 'XOM',  'INTC', 'AMD',   'CSCO', 'PFE',  'WMT',  'CRM',
    'ABT',  'CVX',  'NKE',  'MRK',   'COST', 'ACN',  'LLY',  'TMO',
    'ABBV', 'AVGO', 'QCOM', 'TXN',   'NEE',  'HON',  'MDT',  'UNP',
    'LOW',  'PM',   'UPS',  'BMY',   'GS',   'MS',   'BLK',  'SCHW',
    'SPGI', 'ICE',
]

START_DATE = '2022-07-01'
END_DATE   = '2023-12-31'

CACHE_PATH = '../data/raw/alpaca_minute_data.parquet'

print(f'Universe: {len(SYMBOLS)} symbols | {START_DATE} → {END_DATE}')
print(f'Cache path: {CACHE_PATH}')


In [ ]:
# === 2. Fetch Minute Bars from Alpaca (with local cache) ===

os.makedirs(os.path.dirname(CACHE_PATH), exist_ok=True)

if os.path.exists(CACHE_PATH):
    print(f'Loading from cache: {CACHE_PATH}')
    result_df = pd.read_parquet(CACHE_PATH)
    print(f'✅ Cache loaded. {len(result_df):,} rows, '
          f'{result_df["order_book_id"].nunique()} symbols')
else:
    print('No cache found — fetching from Alpaca (this may take several minutes)...')
    req = StockBarsRequest(
        symbol_or_symbols=SYMBOLS,
        timeframe=TimeFrame.Minute,
        start=START_DATE,
        end=END_DATE,
        feed='iex',          # free IEX feed; change to 'sip' on paid plan
    )
    bars = client.get_stock_bars(req)
    raw = bars.df.reset_index()

    raw = raw.rename(columns={'symbol': 'order_book_id', 'timestamp': 'datetime'})

    # Convert UTC-aware timestamps to Eastern, then strip timezone
    if raw['datetime'].dt.tz is not None:
        raw['datetime'] = (raw['datetime']
                           .dt.tz_convert('America/New_York')
                           .dt.tz_localize(None))

    # 'money' = turnover proxy (vwap * volume)
    vwap_col = 'vwap' if 'vwap' in raw.columns else 'close'
    raw['money'] = raw[vwap_col] * raw['volume']

    result_df = raw[[
        'order_book_id', 'datetime',
        'open', 'high', 'low', 'close', 'volume', 'money',
    ]].copy()

    # Keep only regular-hours bars (09:30–16:00 Eastern)
    t = result_df['datetime'].dt.time
    import datetime as _dt
    result_df = result_df[
        (t >= _dt.time(9, 30)) & (t <= _dt.time(16, 0))
    ].copy()

    result_df.sort_values(['order_book_id', 'datetime'], inplace=True)
    result_df.reset_index(drop=True, inplace=True)

    result_df.to_parquet(CACHE_PATH, index=False)
    print(f'✅ Fetched and cached {len(result_df):,} rows to {CACHE_PATH}')

result_df['datetime'] = pd.to_datetime(result_df['datetime'])
result_df['date']     = result_df['datetime'].dt.date

print(f'\nDataset summary:')
print(f'  Total rows : {len(result_df):,}')
print(f'  Symbols    : {result_df["order_book_id"].nunique()}')
print(f'  Date range : {result_df["date"].min()} → {result_df["date"].max()}')
print(result_df.head())


In [ ]:
# === 3. Feature Engineering: Daily Return Rate ===

if result_df is not None:
    print("--- Calculating Daily Return Rate ---")
    
    # --- Get Daily Closing Prices ---
    # Group by stock and date, then take the last known price of the day as the daily close.
    daily_close = result_df.groupby(['order_book_id', 'date'])['close'].last().reset_index()
    daily_close.rename(columns={'close': 'daily_close'}, inplace=True)
    print("✅ Calculated daily closing prices.")

    # --- Pivot to Wide Format ---
    # Reshape the data so that rows are dates and columns are stock IDs.
    # This format is essential for most quantitative analysis.
    daily_close_wide = daily_close.pivot(index='date', columns='order_book_id', values='daily_close')
    print("✅ Pivoted daily close data to wide format (dates x stocks).")

    # --- Calculate Daily Percentage Change (Return) ---
    # The pct_change() function calculates the return from the previous day's close.
    # fill_method='ffill' is used to forward-fill missing values before calculation to avoid excessive NaNs.
    daily_ret = daily_close_wide.pct_change().fillna(0)
    print("✅ Calculated daily percentage returns.")

    # --- Display Result ---
    print("\n--- Daily Return Data (Head) ---")
    print(daily_ret.head())
    
else:
    print("❌ Cannot calculate daily return rate because the base DataFrame is not available.")


In [ ]:
# === 4. Feature Engineering: Daily Turnover Rate ===

if result_df is not None:
    print("--- Calculating Daily Turnover Rate ---")
    
    # --- Aggregate Daily Volume and Money ---
    # Group by stock and date, then sum the volume and money traded for each day.
    daily_agg = result_df.groupby(['order_book_id', 'date']).agg(
        total_volume=('volume', 'sum'),
        total_money=('money', 'sum')
    ).reset_index()
    print("✅ Aggregated daily total volume and money.")

    # --- Pivot to Wide Format ---
    # Create separate wide-format DataFrames for daily volume and money.
    daily_volume_wide = daily_agg.pivot(index='date', columns='order_book_id', values='total_volume')
    daily_money_wide = daily_agg.pivot(index='date', columns='order_book_id', values='total_money')
    print("✅ Pivoted daily volume and money data to wide format.")

    # --- Calculate Daily Turnover Rate ---
    # Turnover rate is calculated as: Daily Volume / (Previous Day's Volume)
    # This measures the change in trading activity.
    daily_turnover = (daily_volume_wide / daily_volume_wide.shift(1)).fillna(0)
    
    # Replace infinite values that can occur from division by zero with NaN, then fill with 0
    daily_turnover.replace([np.inf, -np.inf], np.nan, inplace=True)
    daily_turnover.fillna(0, inplace=True)
    print("✅ Calculated daily turnover rate.")
    
    # --- Display Result ---
    print("\n--- Daily Turnover Rate Data (Head) ---")
    print(daily_turnover.head())
    
else:
    print("❌ Cannot calculate turnover rate because the base DataFrame is not available.")


In [ ]:
# === 5. Feature Engineering: Daily VWAP ===

if result_df is not None:
    print("--- Calculating Daily VWAP (Volume-Weighted Average Price) ---")
    
    # --- Calculate VWAP ---
    # VWAP is a more representative daily price than the simple close price.
    # It's calculated as: Sum(Close Price * Volume) / Sum(Volume) for each day.
    
    # First, calculate 'close' * 'volume' for each minute record.
    result_df['close_vol'] = result_df['close'] * result_df['volume']
    
    # Then, group by stock and date and sum the 'close_vol' and 'volume'.
    vwap_cal = result_df.groupby(['order_book_id', 'date']).agg(
        total_close_vol=('close_vol', 'sum'),
        total_volume=('volume', 'sum')
    ).reset_index()
    
    # Finally, compute VWAP.
    vwap_cal['vwap'] = vwap_cal['total_close_vol'] / vwap_cal['total_volume']
    print("✅ Calculated daily VWAP values.")

    # --- Pivot to Wide Format ---
    vwap_wide = vwap_cal.pivot(index='date', columns='order_book_id', values='vwap').fillna(0)
    print("✅ Pivoted VWAP data to wide format.")
    
    # --- Calculate VWAP Daily Return ---
    # Calculate the daily percentage change of the VWAP.
    vwap1pct = vwap_wide.pct_change().fillna(0)
    print("✅ Calculated daily percentage returns based on VWAP.")

    # --- Display Result ---
    print("\n--- Daily VWAP Return Data (Head) ---")
    print(vwap1pct.head())

else:
    print("❌ Cannot calculate VWAP because the base DataFrame is not available.")


In [ ]:
# === 6. Barra Factor Integration: Size Factor ===
# Barra risk factor data is not available via the Alpaca API.
# Setting barra_size to None so the save step handles it gracefully.
# If you have access to Barra/FactSet data, load and pivot it here.

barra_size = None
print('ℹ️  Barra size factor skipped (no source available via Alpaca).')
print('   Downstream notebooks will continue without it.')


In [ ]:
# === 7. Save All Processed Data ===

print("--- Saving all processed data to files ---")

# --- Create Output Directory ---
output_dir = '../data/processed/wide_data_preparation'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"✅ Created directory: {output_dir}")

# --- Define a saving function for consistency ---
def save_dataframe(df, name, directory):
    """Saves a DataFrame to a CSV file if it exists."""
    if df is not None:
        file_path = os.path.join(directory, f"{name}.csv")
        try:
            df.to_csv(file_path)
            print(f"✅ Successfully saved '{name}' to {file_path}")
        except Exception as e:
            print(f"❌ Error saving '{name}': {e}")
    else:
        print(f"⚠️ Skipping '{name}' because it was not generated.")

# --- Save all generated features ---
save_dataframe(daily_ret, 'daily_ret_data', output_dir)
save_dataframe(daily_turnover, 'daily_turnover_data', output_dir)
save_dataframe(daily_money_wide, 'daily_money_data', output_dir)
save_dataframe(vwap_wide, 'vwap_daily_data', output_dir)
save_dataframe(vwap1pct, 'vwap1pct_daily_data', output_dir)
save_dataframe(daily_close_wide, 'daily_close_data', output_dir)
save_dataframe(barra_size, 'barra_size_data', output_dir)

print("\n--- Data Preparation and Feature Engineering Complete ---")
